# CEPHA31 — add APOcc + PPOcc (pseudo-label PoC)

Fine-tunes the 29-landmark CEPHA29 model into a **31-landmark** model by adding the
two occlusal-plane points, then exports web-ready ONNX and the SSM prior.

**Why pseudo-labels?** `APOcc` / `PPOcc` are *constructed* points, so their
training labels are derived from the existing 29:
- `APOcc = midpoint(UIT, LIT)` (incisor overbite)
- `PPOcc = midpoint(UMT, LMT)` (molar contact)

This validates the full pipeline (patch → data → train → export → app). It is **not**
a claim that the learned occlusal points are clinically accurate — real accuracy
needs human annotations (the next phase).

**Kaggle setup**
1. Accelerator: **GPU T4 x2** (one is used).
2. Internet: **On** (dataset + timm weights).
3. *Optional, recommended:* attach the original 29-landmark `best.pt` as a Kaggle
   dataset — the notebook auto-detects it and fine-tunes from it instead of ImageNet.
4. **Save & Run All**, then download the `deliverables/` folder.

Outputs: `landmarks-cepha31.onnx`, `landmarks-cepha31.int8.onnx`,
`landmarks-cepha31-verify.png`, `shapeModel.cepha31.json`, `cepha31-ood.csv`.

In [ ]:
!pip -q install timm albumentations opencv-python-headless pyyaml onnx onnxruntime pillow numpy scipy pandas
print("deps installed")

In [ ]:
import os
%cd /kaggle/working
!rm -rf CephaloHRNet CephalometryStudio
!git clone --depth 1 https://github.com/Cestovatels/CephaloHRNet.git
!git clone --depth 1 https://github.com/Muhammad-Nabeel-Sh/CephalometryStudio.git
assert os.path.exists("CephaloHRNet/models/hrnet.py"), "CephaloHRNet clone failed"
app_export = "CephalometryStudio/scripts/export-landmark-onnx.py"
assert os.path.exists(app_export), "app clone failed"
assert "CEPHA31" in open(app_export, encoding="utf-8").read(), \
    "app clone lacks the cepha31 work — push the repo first"
print("cloned; cepha31 support present")

In [ ]:
import os, hashlib, zipfile, requests

def find_dataset(root):
    for base, dirs, _ in os.walk(root):
        for split in ("train", "valid"):
            if os.path.isdir(os.path.join(base, split, "Cephalograms")):
                return base
    return None

DATASET = find_dataset("/kaggle/input")
if DATASET is None:
    URL = "https://ndownloader.figshare.com/files/51041642"
    MD5 = "e0bd645bca6759abdae4f199d841bda6"
    SIZE = 2098209792
    ZIP = "/kaggle/working/Aariz.zip"
    if not os.path.exists(ZIP) or os.path.getsize(ZIP) != SIZE:
        print("downloading Aariz.zip (~1.95 GB)…")
        with requests.get(URL, stream=True, headers={"User-Agent": "Mozilla/5.0"}) as r:
            r.raise_for_status()
            with open(ZIP, "wb") as f:
                for chunk in r.iter_content(1 << 20):
                    f.write(chunk)
    h = hashlib.md5()
    with open(ZIP, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == MD5, "Aariz.zip md5 mismatch — delete and re-run"
    with zipfile.ZipFile(ZIP) as z:
        z.extractall("/kaggle/working/aariz")
    DATASET = find_dataset("/kaggle/working/aariz")
print("DATASET =", DATASET)
assert DATASET, "dataset not found (need Internet On or an attached Kaggle dataset)"

In [ ]:
import os, glob
SPLITS = {}
for split in ("train", "valid"):
    imgs = os.path.join(DATASET, split, "Cephalograms")
    ann_root = os.path.join(DATASET, split, "Annotations", "Cephalometric Landmarks")
    jr = [d for d in glob.glob(os.path.join(ann_root, "*")) if "Junior" in d]
    sr = [d for d in glob.glob(os.path.join(ann_root, "*")) if "Senior" in d]
    SPLITS[split] = {"images": imgs, "junior": jr[0] if jr else None, "senior": sr[0] if sr else None}
    print(split, "images:", len(glob.glob(os.path.join(imgs, "*"))),
          "| junior:", SPLITS[split]["junior"], "| senior:", SPLITS[split]["senior"])
assert SPLITS["train"]["senior"] and SPLITS["valid"]["senior"], "annotations not found"

In [ ]:
import re

NEW_SYMBOLS = ["APOcc", "PPOcc"]
TITLES = {"APOcc": "Anterior Occlusal Point", "PPOcc": "Posterior Occlusal Point"}

# ── data/dataset.py: extend LANDMARK_SYMBOLS + NUM_LANDMARKS + titles ──
dp = "CephaloHRNet/data/dataset.py"
src = open(dp, encoding="utf-8").read()
base = re.search(r"LANDMARK_SYMBOLS = \[(.*?)\]", src, re.S).group(1)
syms = re.findall(r'''['"]([^'"]+)['"]''', base)
assert len(syms) == 29, ("expected 29 base symbols, got", syms)
all_syms = syms + NEW_SYMBOLS
new_block = "LANDMARK_SYMBOLS = [\n" + "".join("    %r,\n" % s for s in all_syms) + "]"
src = re.sub(r"LANDMARK_SYMBOLS = \[.*?\]", new_block, src, count=1, flags=re.S)
src = re.sub(r"NUM_LANDMARKS\s*=\s*29", "NUM_LANDMARKS = 31", src, count=1)
src = src.replace("IMG_EXTS = [", "LANDMARK_TITLES.update(%r)\nIMG_EXTS = [" % TITLES, 1)
open(dp, "w", encoding="utf-8").write(src)

# ── train.py: build the 31-channel head ──
tp = "CephaloHRNet/train.py"
src = open(tp, encoding="utf-8").read()
assert "num_landmarks=29" in src
src = src.replace("num_landmarks=29", "num_landmarks=31")
open(tp, "w", encoding="utf-8").write(src)

# verify by text (no import side effects)
chk = open(dp, encoding="utf-8").read()
assert "NUM_LANDMARKS = 31" in chk and "APOcc" in chk and "PPOcc" in chk
print("patched dataset.py + train.py ->", len(all_syms), "landmarks")

In [ ]:
import os, json, glob
import numpy as np

def read_symbols(path):
    data = json.load(open(path, encoding="utf-8"))
    out = {}
    for lm in data.get("landmarks", []):
        v = lm.get("value") or {}
        if lm.get("symbol") and "x" in v and "y" in v:
            out[lm["symbol"]] = (float(v["x"]), float(v["y"]))
    return out

def averaged(dirs, name):
    sets = [read_symbols(os.path.join(d, name)) for d in dirs
            if os.path.exists(os.path.join(d, name))]
    if not sets:
        return None
    keys = set(sets[0])
    for s in sets[1:]:
        keys &= set(s)
    return {k: (float(np.mean([s[k][0] for s in sets])),
                float(np.mean([s[k][1] for s in sets]))) for k in keys}

def mid(a, b):
    return ((a[0] + b[0]) / 2.0, (a[1] + b[1]) / 2.0)

written = 0
for split, info in SPLITS.items():
    dirs = [d for d in (info["junior"], info["senior"]) if d]
    out_dir = os.path.join(DATASET, split, "Annotations", "Cephalometric Landmarks", "Pseudo31")
    os.makedirs(out_dir, exist_ok=True)
    for p in sorted(glob.glob(os.path.join(dirs[0], "*.json"))):
        name = os.path.basename(p)
        avg = averaged(dirs, name)
        if not avg or not all(s in avg for s in ("UIT", "LIT", "UMT", "LMT")):
            continue
        avg["APOcc"] = mid(avg["UIT"], avg["LIT"])
        avg["PPOcc"] = mid(avg["UMT"], avg["LMT"])
        doc = {"ceph_id": os.path.splitext(name)[0],
               "landmarks": [{"symbol": s, "value": {"x": round(v[0], 3), "y": round(v[1], 3)}}
                             for s, v in avg.items()]}
        json.dump(doc, open(os.path.join(out_dir, name), "w"))
        written += 1
print("wrote", written, "Pseudo31 annotation files (all splits)")

In [ ]:
import glob, os, shutil
# A few real cephalograms for INT8 calibration + the verify overlay.
imgs = [f for f in sorted(glob.glob(os.path.join(SPLITS["train"]["images"], "*")))
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"))][:16]
assert imgs, "no training images found"
os.makedirs("/kaggle/working/calib", exist_ok=True)
for f in imgs:
    shutil.copy(f, "/kaggle/working/calib/" + os.path.basename(f))
print("calib images:", len(imgs), "| sample:", os.path.basename(imgs[0]))

In [ ]:
import glob, subprocess, sys

# Fine-tune from an attached 29-landmark best.pt if present; else ImageNet init.
found = [p for p in glob.glob("/kaggle/input/**/*.pt", recursive=True) if p.endswith("best.pt")]
init = ["--pretrained-path", found[0]] if found else ["--pretrained"]
print("init:", found[0] if found else "ImageNet (timm --pretrained)")

cmd = [sys.executable, "train.py",
       "--data", DATASET, "--model", "hrnet_w32",
       "--imgsz", "768", "--batch", "4", "--epochs", "200",
       "--loss", "awing", "--amp", "--device", "0",
       "--workers", "2", "--patience", "30",
       "--project", "runs/train", "--name", "cepha31_w32_768",
       "--annotators", "Pseudo31"] + init
print(" ".join(cmd))
subprocess.run(cmd, cwd="/kaggle/working/CephaloHRNet", check=True)

In [ ]:
import glob, os, subprocess, sys
BEST = sorted(glob.glob("/kaggle/working/CephaloHRNet/runs/train/cepha31_*/best.pt"),
              key=os.path.getmtime)[-1]
SAMPLE = sorted(glob.glob("/kaggle/working/calib/*"))[0]
print("weights:", BEST)
cmd = [sys.executable, "CephalometryStudio/scripts/export-landmark-onnx.py",
       "--repo", "/kaggle/working/CephaloHRNet",
       "--weights", BEST, "--model", "hrnet_w32",
       "--num-landmarks", "31", "--input-size", "768",
       "--out", "/kaggle/working/landmarks-cepha31.onnx",
       "--quantize", "--calib-dir", "/kaggle/working/calib",
       "--verify", SAMPLE]
print(" ".join(cmd))
subprocess.run(cmd, cwd="/kaggle/working", check=True)

In [ ]:
import subprocess, sys, os
# SSM prior for the 31-landmark set (built from the Pseudo31 training annotations).
subprocess.run([sys.executable, "CephalometryStudio/scripts/build-shape-model.py",
                "--set", "cepha31",
                "--ann-dir", os.path.join(DATASET, "train", "Annotations", "Cephalometric Landmarks", "Pseudo31"),
                "--k", "10",
                "--out", "/kaggle/working/shapeModel.cepha31.json"],
               cwd="/kaggle/working", check=True)

In [ ]:
import subprocess, sys, os
# Confidence / shape-residual distribution for the new set (calibrates the gate).
subprocess.run([sys.executable, "CephalometryStudio/scripts/eval-landmark-model.py",
                "--model", "/kaggle/working/landmarks-cepha31.int8.onnx",
                "--set", "cepha31", "--limit", "200", "--dark",
                "--ann-dir", os.path.join(DATASET, "valid", "Annotations", "Cephalometric Landmarks", "Pseudo31"),
                "--images", os.path.join(DATASET, "valid", "Cephalograms"),
                "--ssm", "/kaggle/working/shapeModel.cepha31.json",
                "--report-ood", "--ood-out", "/kaggle/working/cepha31-ood.csv"],
               cwd="/kaggle/working", check=True)

In [ ]:
import glob, hashlib, os, shutil
out = "/kaggle/working/deliverables"
os.makedirs(out, exist_ok=True)
for name in ("landmarks-cepha31.onnx", "landmarks-cepha31.int8.onnx",
             "landmarks-cepha31-verify.png", "shapeModel.cepha31.json",
             "cepha31-ood.csv"):
    for p in glob.glob("/kaggle/working/" + name):
        shutil.copy(p, out)

def sha(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()

print("=== DELIVERABLES (download this folder) ===")
for f in sorted(os.listdir(out)):
    p = os.path.join(out, f)
    print(f"{os.path.getsize(p)/1e6:9.2f} MB  {f}\n            sha256 {sha(p)}")

## Next

1. Download the `deliverables/` folder.
2. Inspect `landmarks-cepha31-verify.png`: every dot is labelled `index:symbol`;
   confirm **29:APOcc** and **30:PPOcc** land on the occlusal plane and that
   indices 0–28 are unchanged from CEPHA29. This is the channel-order guard.
3. Place `landmarks-cepha31.int8.onnx` in `public/models/` and
   `shapeModel.cepha31.json` in `src/data/`.
4. We then wire up the `cepha31` landmark set in the app (manifest + prior) and
   recalibrate the quality gate from `cepha31-ood.csv`.
5. Later (real value): annotate `Ba/Pr/Id` by hand with the in-app
   "Export AI-training landmarks" feature and fine-tune again.